# Fine-mapping and TWAS analysis

This mini-protocol selects among univariate, multivariate, multigene, functional and summary-statistic fine-mapping routes according to the available data and analysis goal.

#### Miniprotocol Timing

Timing: TBD

## Overview

The [MNM regression module](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html) analyzes individual-level genotype and molecular-phenotype data. `qtl_dataset_construct+susie_twas` performs univariate SuSiE fine-mapping and estimates TWAS weights; `mnm` jointly analyzes multiple molecular traits with multivariate priors; `mnm_genes` extends the multivariate model across genes; and `fsusie` fine-maps functional or epigenomic phenotypes.

The [RSS module](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/rss_analysis.html) instead combines GWAS summary statistics with an external LD reference. These five commands are alternative analysis routes rather than a mandatory chain.

## Steps

| **Analysis goal** | **Commands to run, in order** | **Inputs** |
| --- | --- | --- |
| Univariate molecular-QTL fine-mapping and TWAS weights | 1 | Genotype BED set, phenotype manifest, covariates and association windows under `input/colocboost/` |
| Multivariate fine-mapping across molecular traits | 2 | Step 1 inputs plus `tests/fixtures/qtl_mini/fine_mapping_meta.tsv` |
| Multigene multivariate fine-mapping | 3 | Multivariate inputs plus phenotype-ID map and retained-sample list under `input/colocboost/` |
| Fine-map functional or epigenomic phenotypes | 4 | `tests/fixtures/qtl_mini/protocol_example.genotype.chr22.bed`; phenotype manifest, covariates and windows |
| GWAS summary-statistic fine-mapping | 5 | `tests/fixtures/rss_analysis/protocol_example.rss_mwe.gwas_meta.tsv`; `input/ld_reference/protocol_example.ld_meta_file.tsv` |

Run only the command matching the analysis goal.

### [1. Univariate fine-mapping and TWAS](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html)

**What it does:** `qtl_dataset_construct+susie_twas` builds the regional dataset, fits SuSiE and saves fine-mapping results and cross-validated TWAS weights.

In [ ]:
sos run pipeline/mnm_regression.ipynb qtl_dataset_construct+susie_twas   --name protocol_example --cwd output/mnm/univariate   --genoFile tests/fixtures/qtl_mini/protocol_example.genotype.chr22.bed   --phenoFile tests/fixtures/qtl_mini/protocol_example.pheno_manifest_context.tsv   --covFile tests/fixtures/qtl_mini/example_covariates.tsv   --customized-association-windows tests/fixtures/qtl_mini/association_windows.bed   --region-name ENSG00000130538 --transpose-covariates --save-data -j1

### [2. Multivariate fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html)

**What it does:** `mnm` jointly fine-maps multiple molecular traits using the analyses and prior settings specified by the fine-mapping metadata.

In [ ]:
sos run pipeline/mnm_regression.ipynb mnm   --name protocol_example --cwd output/mnm/multivariate   --genoFile tests/fixtures/qtl_mini/protocol_example.genotype.chr22.bed   --phenoFile tests/fixtures/qtl_mini/protocol_example.pheno_manifest_context.tsv   --covFile tests/fixtures/qtl_mini/example_covariates.tsv   --customized-association-windows tests/fixtures/qtl_mini/association_windows.bed   --fine-mapping-meta tests/fixtures/qtl_mini/fine_mapping_meta.tsv   --transpose-covariates --save-data -j1

### [3. Multigene multivariate fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html)

**What it does:** `mnm_genes` coordinates multivariate fine-mapping across genes while preserving phenotype identifiers and a common retained-sample set.

In [ ]:
sos run pipeline/mnm_regression.ipynb mnm_genes   --name protocol_example --cwd output/mnm/multigene   --genoFile tests/fixtures/qtl_mini/protocol_example.genotype.chr22.bed   --phenoFile tests/fixtures/qtl_mini/protocol_example.pheno_manifest_context.tsv   --covFile tests/fixtures/qtl_mini/example_covariates.tsv   --customized-association-windows tests/fixtures/qtl_mini/association_windows.bed   --pheno-id-map-file tests/fixtures/qtl_mini/pheno_id_map.tsv   --fine-mapping-meta tests/fixtures/qtl_mini/fine_mapping_meta.tsv   --keep-samples <path/to/keep_samples.txt> --save-data -j1

### [4. Functional fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/mnm_regression.html)

**What it does:** `fsusie` applies functional SuSiE to epigenomic or other functional phenotypes and saves posterior fine-mapping results and residual data.

In [ ]:
sos run pipeline/mnm_regression.ipynb fsusie   --name protocol_example --cwd output/mnm/fsusie   --genoFile tests/fixtures/qtl_mini/protocol_example.genotype.chr22.bed   --phenoFile tests/fixtures/qtl_mini/pheno_manifest.tsv   --covFile tests/fixtures/covariate_hidden_factor/covariates.tsv   --customized-association-windows tests/fixtures/qtl_mini/association_windows.bed   --cis-window 0 --max-cv-variants 5000 --save-data -j1

### [5. Summary-statistic fine-mapping](https://statfungen.github.io/xqtl-protocol/code/mnm_analysis/mnm_methods/rss_analysis.html)

**What it does:** the RSS chain harmonizes regional GWAS summary statistics with the LD reference, runs SuSiE-RSS fine-mapping and produces a regional diagnostic plot.

In [ ]:
sos run pipeline/rss_analysis.ipynb   generate_manifest+generate_gwas_sumstats+gwas_fine_mapping+gwas_rss_plot   --cwd output/mnm/rss --modular-script-dir code/script   --gwas-meta tests/fixtures/rss_analysis/protocol_example.rss_mwe.gwas_meta.tsv   --regions chr22:49355984-50799822   --ld-meta tests/fixtures/ld_reference/ld_meta_file.tsv

## Output Files

| Route | Output filename and relative path | Description |
| --- | --- | --- |
| Univariate | `output/mnm/univariate/**/*.univariate_bvsr.rds`; `output/mnm/univariate/**/*.twas_weights.rds` | SuSiE posterior and TWAS-weight objects |
| Multivariate | `output/mnm/multivariate/**/*.multivariate_bvsr.rds`; `output/mnm/multivariate/**/*.twas_weights.rds` | Joint multivariate posterior and weights |
| Multigene | `output/mnm/multigene/**/*` | Per-region, per-gene multivariate results and metadata |
| Functional | `output/mnm/fsusie/**/*.fsusie.rds`; saved residual objects | Functional fine-mapping posterior and residual data |
| RSS | `output/mnm/rss/fine_mapping/AD_Bellenguez_2022.chr22_49355984_50799822.gwas_finemap.rds`; regional plot under `output/mnm/rss/plots/` | Summary-statistic fine-mapping result and diagnostic plot |

## Anticipated Results

Each route produces posterior inclusion probabilities and credible-set information for the selected region and analysis model. Routes that estimate prediction weights also save TWAS-weight objects. The RSS route additionally records the harmonized regional summary statistics and LD-based diagnostics.

## Command interface

In [ ]:
sos run pipeline/mnm_regression.ipynb -h

In [ ]:
sos run pipeline/rss_analysis.ipynb -h